# Finding $\Gamma^{\max}$ via the big-M heuristic

This notebook implements the _linearized_ model from [1, Sec. II.C].

It is known that the single-level model gives the same results.

The purpose of this notebook is to implement the big-M
heuristic for the bilevel model.

Specifically $\Gamma^{\max}$ is not large enough if either:
- there is a $k$ with $x_k = 1$ for which $|\phi_k(w) - \phi_k^{-}(w)| \approx \Gamma^{\max}$ 

**or** 

- there is a $k$ with $x_k = 0$ and $|\phi_k^{-}(w)| \approx \Gamma^{\max}$.

### Alnowibet market power index
As an added bonus, not discussed in the main text of the dissertation, this
notebook also calculates the value of a market power index (the average nodal
price deviation index) used in [2].

## References
[1] Garces, L.P et al. “A Bilevel Approach to Transmission Expansion Planning Within a Market Environment.” *IEEE Transactions on Power Systems* 24.3 (2009): 1513–1522.


[2] Alnowibet, Khalid A, Ahmad M Alshamrani, and Adel F Alrasheedi. “A Bilevel Stochastic Optimization Framework for Market-Oriented Transmission Expansion Planning Considering Market Power.” *Energies* 16.7 (2023).


In [ ]:
# IMPORTS
# --------

# Standard libraries
import importlib
import pandas as pd
import numpy as np

import networkx as nx

# Optimizer
import xpress as xp
# xp.init('C:/Program Files/xpressmp/bin/xpauth_personal.xpr') # for local version
xp.init('C:/Program Files/xpressmp/bin/xpauth_uoe2.xpr')

# Data
import garces_data_module as data

# Model
import garces_model as model

importlib.reload(data)
importlib.reload(model)

<module 'garces_et_al_model_7d' from 'c:\\Users\\ajmac\\Documents\\ORMSc\\Code\\Required\\garces_et_al_model_7d.py'>

In [2]:
# DATA AND MODEL PARAMETERS
# -------------------------

data_parameters = {"Gamma_max": 1, # when =1 doesn't find optimal solution.
                #    "PDI_max": 0.001,
                   "c_max": 30_000_000
                   }

data.load_data(parameters = data_parameters)
# data.load_data(scenario_list=[1])

# Force particular lines to be open
# PICK ONE OF THE FOLLOWING LINES, comment out the others:
new_lines = [] # This doesn't all new lines to close; it doesn't force anything.
# new_lines = [9,21,24,26] # min cost scenario 1 GARCES
# new_lines = [9,24,26,41] # min cost scenario 1 according to AJM (garces_et_al_02d.ipynb)
# new_lines = [14,21,26,29,44] # min cost scenario 2 GARCES
# new_lines = [9,14,26,29] # min cost scenario 3 GARCES

# "bilevel = True" and "alternative_objective = False" is the vanilla Garces:
bilevel = True # Has to be False if particular lines are forced open.
alternative_objective = False
alnowibet = 2 # set to 1 for PDI constraint, 2 for NUI constraint

model_params = {"new_lines": new_lines, 
                "bilevel"  : bilevel,
                "alternative_objective" : alternative_objective,
                "alnowibet": alnowibet # Used to add an Alnowibet market power constraint
                }

solver_options = {"outputlog": 0}

# sol_dict = model.run_model(data.DataStore, model_params=model_params, solver_options=solver_options)

In [3]:
failed_Gamma_max = True

while failed_Gamma_max:

    sol_dict = model.run_model(data.DataStore, model_params=model_params, solver_options=solver_options)

    SW = {k: data.sigma*data.delta[k]*v for k,v in sol_dict['sw_ls'].items()}
    benefit = sum(SW.values()) - sol_dict['tics']

    print("")
    print(f"Gamma_max: {data.Gamma_max}")
    print(f"Benefit: {benefit}")

    # Here is the key idea: check if the problematic quantities are too close to
    # Gamma_max
    x_sol = sol_dict['x']
    phi_sol = sol_dict['phi']
    phi_minus_sol = sol_dict['phi_minus']

    failed_Gamma_max = False
    for k in data.lines:
        if x_sol[k] > 0.99:
            quant_1 = max([abs(phi_sol[w,k] - phi_minus_sol[w,k]) for w in data.scenarios])
            if quant_1 * 10 > data.Gamma_max:
                failed_Gamma_max = True
                print(f'Gamma_max = {data.Gamma_max:.2f} is too small.')
                break

        else:
            quant_2 = max([abs(phi_minus_sol[w,k]) for w in data.scenarios])
            if quant_2 * 10 > data.Gamma_max:
                failed_Gamma_max = True
                print(f'Gamma_max = {data.Gamma_max:.2f} is too small.')
                break
    
    if failed_Gamma_max:
        # Update the value of Gamma_max <- slightly annoying that this currently
        # requires reloading the entire data. REWORK.
        data_parameters['Gamma_max'] = data.Gamma_max * 10
        data.load_data(parameters=data_parameters)
        print(f"Gamma_max set to {data.Gamma_max:.2f}") 


Gamma_max: 1
Benefit: 263753399.99999994
Gamma_max = 1.00 is too small.
Gamma_max set to 10.00

Gamma_max: 10
Benefit: 267613685.9999994
Gamma_max = 10.00 is too small.
Gamma_max set to 100.00

Gamma_max: 100
Benefit: 267613686.0


In [4]:
sol_dict = model.run_model(data.DataStore, model_params=model_params, solver_options=solver_options)

x_sol = sol_dict['x'] # get the lines now open
soln_line_list = [k for k,v in x_sol.items() if v >=0.99]

new_lines = [k for k in soln_line_list if k not in data.existing_lines]

# Print the new lines we build:
print("NEW LINES:")
print("----------")
for k in new_lines: # If the line is open...
    print(f"Line {k}: ({data.o[k]}-{data.r[k]})") # ... list it here!

# Print the total investment cost:
print("")
print(f"Total Investment Cost : EUR {sol_dict['tics']/10**6:.2f} million")

# If the plan has been based on a single scenario, print the social welfare in
# that specific scenario:
if len(data.scenarios) == 1:
    print("")
    print(f"Social Welfare: EUR {data.sigma*sum(sol_dict['sw_ls'].values())/10**6:.2f} million")

# Get the total load shed:
print("")
print(f"Total load shed: {sum(sol_dict['r'].values())} MW")

# Calculate the average social welfare ACROSS SCENARIOS for those fixed lines:
data.load_data(parameters=data_parameters)

# Reset the model parameters. In particular make sure that we force the lines
# to be open that were determined in the plan above.
model_params["new_lines"] = new_lines

solver_options = {"outputlog": 0}

sol_dict = model.run_model(data.DataStore, model_params=model_params, solver_options=solver_options)


sw = {k: data.sigma*v/10**6 for k,v in sol_dict['sw_ls'].items()}
print("")
for w in data.scenarios:
    print(f"Social Welfare in Scenario {w}: {sw[w]:.2f}")

SW = {k: data.sigma*data.delta[k]*v for k,v in sol_dict['sw_ls'].items()}

print("")
print(f"Average Social Welfare: EUR {sum(SW.values())/10**6:.2f} million")
print("")
print(f"    *N.B. Scenarios weights are: {list(data.delta.values())}*")

if bilevel:
    # Calculate the Alnowibet PDI market power index:
    lambda_s_bar_sol = sol_dict["lambda_s_bar"]
    lambda_bar_sol = sol_dict["lambda_bar"]
    # Get the scenario-weighted LMP variation:
    LMP_variation_list = [abs(lambda_s_bar_sol[s]-lambda_bar_sol) for s in data.buses]
    PDI = sum(LMP_variation_list) / (len(data.buses) * lambda_bar_sol)
    print("")
    print(f"Alnowibet PDI: {PDI}")


NEW LINES:
----------
Line 14: (4-6)
Line 26: (3-5)
Line 29: (4-6)
Line 36: (2-3)
Line 44: (4-6)

Total Investment Cost : EUR 25.10 million

Total load shed: 0.0 MW

Social Welfare in Scenario 1: 278.49
Social Welfare in Scenario 2: 307.79
Social Welfare in Scenario 3: 306.09

Average Social Welfare: EUR 292.71 million

    *N.B. Scenarios weights are: [0.5, 0.25, 0.25]*

Alnowibet PDI: 0.0022828290966696438
